# 년도별 Oracle Data 하나로 합치기  

In [ ]:
import pandas as pd
import glob
import os

In [ ]:
# =========================================================
# 2. 연도별 데이터 불러오기
# =========================================================
dfs = {}

files = glob.glob("data/*.csv")

for file in files:

    # 파일명에서 연도 추출
    year = os.path.basename(file).split("_")[0]

    # csv 읽기
    df = pd.read_csv(file)

    # dictionary 저장
    dfs[year] = df

    print(year, df.shape)

In [ ]:
# =========================================================
# 3. 각 연도별 컬럼 개수 확인
# =========================================================

print("\n========== 연도별 컬럼 개수 ==========\n")

for year, df in dfs.items():
    print(f"{year}: {len(df.columns)}개")

In [ ]:
# =========================================================
# 4. 모든 연도에 공통으로 존재하는 컬럼 찾기 -> 확인해보니 모든 컬럼들이 다 같음 -> 그냥 concat해서 진행해도 될듯
# =========================================================

common_columns = set(list(dfs.values())[0].columns)

for df in list(dfs.values())[1:]:
    common_columns = common_columns.intersection(df.columns)

common_columns = sorted(list(common_columns))

print("\n========== 공통 컬럼 개수 ==========\n")
print(len(common_columns))


print("\n========== 공통 컬럼 목록 ==========\n")

for col in common_columns:
    print(col)

In [ ]:
# =========================================================
# 5. 연도별로만 존재하는 컬럼 확인
# =========================================================

print("\n========== 연도별 특수 컬럼 ==========\n")

for year, df in dfs.items():

    unique_cols = set(df.columns) - set(common_columns)

    print(f"\n[{year} 전용 컬럼]")

    for col in sorted(unique_cols):
        print(col)


In [ ]:
selected_features = [

    # =========================================================
    # 1. 기본 정보 컬럼
    # =========================================================

    "playername",     # Player : 선수 이름 -> 선수별 집계 및 분석에 필요
    "teamname",       # Team : 팀 이름 -> 팀 정보 확인
    "position",       # Pos : 포지션 -> 포지션별 정규화 수행 예정

    # "champion",     # Champion : 챔피언 이름 -> 메타 영향이 너무 커 제외
    # "event",        # Event : 대회명 -> 현재 LCK만 사용할 예정이라 제외
    # "rank",         # Rank : 솔로랭크 랭킹 -> 프로 경기 분석과 직접 연결 어려움
    # "lp",           # LP : 래더 포인트 -> 솔랭 데이터 성격이라 제외

    "patch",          # 패치 버전 -> 패치별 메타 차이 보정을 위해 사용
    "year",           # 연도 -> 연도별 메타 차이 보정을 위해 사용
    "date",           # 경기 날짜 -> 시간 흐름에 따른 트렌드 분석 가능


    # =========================================================
    # 2. 라인전 체급 관련 컬럼
    # =========================================================

    "golddiffat15",   # GD15 : 15분 골드 차이 -> 라인전 우위 판단 핵심 지표
    "xpdiffat15",     # XPD15 : 15분 경험치 차이 -> 성장 우위 반영
    "csdiffat15",     # CSD15 : 15분 CS 차이 -> 라인전 압박 능력 판단

    # "golddiffat10", # GD10 : 초반 골드 차이 -> GD15와 정보 중복 가능성
    # "golddiffat20", # GD20 : 중후반 영향 포함 -> 라인전 체급 의미 약화

    # "xpdiffat10",   # XPD10 : XPD15와 정보 중복 가능성
    # "xpdiffat20",   # XPD20 : 중후반 영향 포함

    # "csdiffat10",   # CSD10 : CSD15와 정보 중복 가능성
    # "csdiffat20",   # CSD20 : 중후반 영향 포함

    # "gxdiffat15",   # GXD15 : 골드+경험치 통합 지표 -> GD/XPD와 정보 중복 가능성 큼

    "goldat15",       # 15분 골드량 -> 성장 속도 반영
    "xpat15",         # 15분 경험치량 -> 성장량 반영
    "csat15",         # 15분 CS량 -> 안정적 수급 능력 반영

    # "cs%p15",       # CS%P15 : 팀 CS 비중 -> 팀 스타일 영향 큼
    # "lne%",         # LNE% : Lane Control -> 해석 모호 및 중복 가능성


    # =========================================================
    # 3. 전투력 / 캐리력 관련 컬럼
    # =========================================================

    "kills",          # K : 킬 수 -> 캐리력 판단
    "deaths",         # D : 데스 수 -> 안정성 판단
    "assists",        # A : 어시스트 수 -> 교전 기여도 반영

    "teamkills",      # 팀 전체 킬 수 -> KP 계산용으로 사용 예정
    
    # "kd",           # KD : 킬데스 비율 -> KDA와 정보 중복
    # "kpg",          # 경기당 킬 -> 경기 수 영향 큼
    # "apg",          # 경기당 어시스트 -> 경기 수 영향 큼
    # "dpg",          # 경기당 데스 -> 경기 수 영향 큼

    # "ks%",          # 킬 비중 -> 원딜 포지션 편향 심함

    "dpm",            # DPM : 분당 챔피언 피해량 -> 핵심 캐리력 지표
    "damageshare",    # DMG% : 팀 내 딜 비중 -> 팀 내 영향력 판단

    "damagetakenperminute",      # Damage Taken per minute -> 전면 교전 참여도
    "damagemitigatedperminute",  # Damage Mitigated per minute -> 탱커 가치 반영

    # "dmg%p15",      # 15분 이후 딜 비중 -> 후반 메타 영향 큼
    # "d%p15",        # 후반 딜 비중 -> 중복 가능성

    # "ckpm",         # 경기 전체 교전 빈도 -> 개인 역량보다 경기 스타일 영향 큼


    # =========================================================
    # 4. 시야 및 운영 기여 컬럼
    # =========================================================

    "visionscore",    # Vision Score -> 맵 장악 및 운영 능력 판단
    "vspm",           # VSPM : 분당 Vision Score -> 시야 효율 판단

    "wardsplaced",    # 설치 와드 수 -> 시야 확보 능력
    "wardskilled",    # 제거 와드 수 -> 상대 시야 제거 능력

    "wpm",            # WPM : 분당 와드 설치 수 -> 시야 장악 효율
    "wcpm",           # WCPM : 분당 와드 제거 수 -> 시야 제거 효율

    "controlwardsbought", # CWPM 관련 원본 값 -> 운영 기여 반영

    # "wc%",          # 와드 제거율 -> 퍼센트 기반이라 해석 어려움
    # "vwc%",         # visible ward 제거율 -> 세부 의미 약함
    # "iwc%",         # invisible ward 제거율 -> 데이터 안정성 낮음


    # =========================================================
    # 5. 자원 수급 및 성장 효율 컬럼
    # =========================================================

    "earnedgold",         # 획득 골드 -> 실제 성장량 반영
    "earned gpm",         # EGPM : 분당 획득 골드 -> 성장 효율 핵심

    "earnedgoldshare",    # GOLD% : 팀 내 골드 비중 -> 자원 집중도 반영

    # "gpm",              # 분당 골드 -> EGPM과 정보 중복 가능성

    # "gpr",                # GPR : 골드 점유율 -> 자원 우위 판단 -> 11 단계에서 확인 결과 년도별로 존재하지 않는 경우가 있어 제외
    # "gspd",               # GSPD : 골드 소비 차이 -> 자원 활용 효율 -> 11 단계에서 확인 결과 년도별로 존재하지 않는 경우가 있어 제외

    # "total cs",         # 총 CS -> 경기 시간 영향 너무 큼
    "cspm",               # CSPM : 분당 CS -> 안정적 성장 능력

    # "jng%",             # 정글 점유율 -> 정글 포지션 편향 심함


    # =========================================================
    # 6. 탱킹 / 생존 관련 컬럼
    # =========================================================

    # damagetakenperminute 사용
    # damagemitigatedperminute 사용
    # -> 탱커 가치 및 전면 교전 참여도 반영


    # =========================================================
    # 7. 오브젝트 관련 컬럼
    # =========================================================

    # "bn%",             # 바론 점유율 -> 팀 오더 영향 큼
    # "drg%",            # 드래곤 점유율 -> 팀 운영 영향 큼
    # "hld%",            # 전령 점유율 -> 팀 macro 영향 큼
    # "eld%",            # 장로 점유율 -> 경기 상황 영향 큼
    # "grb%",            # 공허 유충 점유율 -> 최신 메타 편향

    # "stl",             # 오브젝트 스틸 -> 표본 수 너무 적음
    # "stlpg",           # 경기당 오브젝트 스틸 -> 표본 수 적음

    # "ft%",             # 첫 타워 비율 -> 팀 전략 영향
    # "fd%",             # 첫 드래곤 비율 -> 팀 전략 영향
    # "fbn%",            # 첫 바론 비율 -> 팀 전략 영향
    # "f3t%",            # 첫 3타워 비율 -> 팀 운영 영향 큼

    # "turretplates",      # PPG : 포탑 방패 획득 -> 초반 압박력 반영 -> 11 단계에서 확인 결과 년도별로 존재하지 않는 경우가 있어 제외
    "damagetotowers",    # TDPG : 타워 피해량 -> 운영 기여도 반영


    # =========================================================
    # 8. 초반 영향력 관련 컬럼
    # =========================================================

    "firstbloodkill",    # First Blood 킬 여부 -> 초반 aggressive 성향
    "firstbloodassist",  # First Blood 어시스트 여부 -> 초반 교전 기여
    "firstbloodvictim",  # First Blood 희생 여부 -> 초반 안정성 판단


    # =========================================================
    # 9. Oracle’s Elixir 자체 평가 지표
    # =========================================================

    "league",            # 현재는 LCK만 필터링에 사용
    "datacompleteness",  # complete 경기만 사용하기 위해 필요
    "gamelength",        # remake 제거를 위한 경기 시간 정보
    "result"             # 승패 분석 및 추가 실험 가능성 고려

    # =========================================================
    # 10. 필터링용 컬럼(다 제외)
    # =========================================================
    # "OE Rating",
    # "OE Grade",
    # "EGR",
    # "MLG"
]

In [ ]:
# =========================================================
# 6. 연도별로만 선택 컬럼만 남기기
# =========================================================
for year in dfs:

    dfs[year] = dfs[year][selected_features]

    print(year, dfs[year].shape)

In [ ]:
# =========================================================
# 7. lck 경기만 남기기
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["league"] == "LCK"
    ]
    print(year, dfs[year].shape)

In [ ]:
# =========================================================
# 8. 불완전한 경기 제거 (값에 decomplete이 들어간 경기 제거)
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["datacompleteness"] == "complete"
    ]
    print(year, dfs[year].shape)

In [ ]:
# =========================================================
# 9. 15분 미만의 비정상 경기 제거 -> 15분 미만의 경기는 경기 도중에 문제가 생겨서 중단된 경기들임. -> 확인 결과 15분 미만의 경기가 없긴 했음. 이상치 없는거 확인. 그리고 lck 내부에서는 정상 경기라면 일단 15분 내의 경기가 불가능.
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["gamelength"] >= 900
    ]
    print(year, dfs[year].shape)

In [ ]:
print(
    dfs[year]["gamelength"].min()
)

In [ ]:
# =========================================================
# 10. 팀 단위 row 제거 -> 선수 개인의 기량만 판단하기 때문에 필요 없다고 판단. -> 각 라인별로의 경기만 남게 됨.
# =========================================================
for year in dfs:
    dfs[year] = dfs[year][
        dfs[year]["position"] != "team"
    ]
    print(year, dfs[year].shape)

In [ ]:
print(
    dfs[year]["position"].unique()
)

In [ ]:
# =========================================================
# 연도별 현재 데이터 개수 확인 (확인용임)
# =========================================================

for year in sorted(dfs.keys()):

    print(
        f"{year} : "
        f"{dfs[year].shape[0]} rows, "
        f"{dfs[year].shape[1]} columns"
    )

In [ ]:
# =========================================================
# 11 .연도별 결측치 확인
# =========================================================
for year in dfs:

    print(f"\n===== {year} =====")

    missing = dfs[year].isnull().sum()

    # 결측치 있는 컬럼만 출력
    missing = missing[missing > 0]

    print(missing)

In [ ]:
# =========================================================
# 모든 시즌에 공통으로 존재하는 컬럼 찾기
# =========================================================
common_columns = None

for year in dfs:

    cols = set(dfs[year].columns)

    if common_columns is None:
        common_columns = cols

    else:
        common_columns = (
            common_columns & cols
        )

print(sorted(common_columns))

In [ ]:
# =========================================================
# selected_features 중 없는 컬럼 확인
# =========================================================

missing_features = []

for col in selected_features:

    if col not in common_columns:

        missing_features.append(col)

print(missing_features)

In [ ]:
# =========================================================
# 연도별 결측치 비율 확인 -> 결측치가 있는 컬럼을 확인했는데 gpr, gspd, turretplates가 100% 였음. 즉 다른 년도에는 존재하지 않는 데이터임 -> selected features에서 제거!
# =========================================================

for year in dfs:

    print(f"\n===== {year} =====")

    missing_ratio = (
        dfs[year].isnull().mean() * 100
    )

    missing_ratio = (
        missing_ratio[missing_ratio > 0]
    )

    print(
        missing_ratio.sort_values(
            ascending=False
        )
    )

In [ ]:
# =========================================================
# 2021 playername 결측 row 확인
# =========================================================

missing_rows = dfs["2021"][
    dfs["2021"]["playername"].isnull()
]

print(missing_rows)

In [ ]:
# 직접 빈 5개의 값을 채워넣으려고 하는데 일단 선수가 누구인지 확인해야함. -> 찾아본 결과 해당 날짜는 구마유시였음.

missing_rows[
    [
        "date",
        "teamname",
        "result",
        "patch"
    ]
]

In [ ]:
# =========================================================
# 확인된 Gumayusi row만 수정
# =========================================================

gumayusi_idx = [
    132663,
    132687,
    132723,
    132747,
    132788
]

dfs["2021"].loc[
    gumayusi_idx,
    "playername"
] = "Gumayusi"

In [ ]:
# 잘 들어갔는지 확인하고
dfs["2021"].loc[
    gumayusi_idx,
    ["playername", "teamname", "position"]
]

In [ ]:
# 다시 결측 확인
dfs["2021"]["playername"].isnull().sum()

In [ ]:
# =========================================================
# 저장 폴더 생성
# =========================================================

os.makedirs(
    "processed",
    exist_ok=True
)

# =========================================================
# 연도별 csv 저장
# =========================================================

for year in dfs:

    save_path = (
        f"processed/{year}_clean.csv"
    )

    dfs[year].to_csv(
        save_path,
        index=False
    )

    print(f"{save_path} 저장 완료")

* 이렇게 전처리를 끝난 년도별 데이터를 아래에서 합쳐주는 작업을 해줌.
```
processed/2014_clean.csv 저장 완료
processed/2015_clean.csv 저장 완료
processed/2016_clean.csv 저장 완료
processed/2017_clean.csv 저장 완료
processed/2018_clean.csv 저장 완료
processed/2019_clean.csv 저장 완료
processed/2020_clean.csv 저장 완료
processed/2021_clean.csv 저장 완료
processed/2022_clean.csv 저장 완료
processed/2023_clean.csv 저장 완료
processed/2024_clean.csv 저장 완료
processed/2025_clean.csv 저장 완료
processed/2026_clean.csv 저장 완료
```
사용한 데이터csv파일은 위와 같음 

# pro로 등록된 선수들만 스크래핑하기 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time
import re

* opgg에서 프로로 등록된 선수들의 riot id를 가지고 오기 위해서 아래와 같은 스크래핑을 진행함 

In [ ]:

driver = webdriver.Chrome()

url = "https://op.gg/ko/lol/spectate/list/pro-gamer?region=kr"

driver.get(url)

time.sleep(5)

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 동적 페이지여서 렌더링이 되야지 버튼이 생기기에 
# 셀레리움으로 코드를 짬 
wait = WebDriverWait(driver, 30)

buttons = driver.find_elements(
    By.XPATH,
    '//*[@id="content-container"]/div[2]/div[2]/main/section/button'
)

print(len(buttons))

In [ ]:
# 버튼이 생기면 클릭해서 다음 페이지로 넘어가게 함
for btn in buttons:

    try:
        driver.execute_script(
            "arguments[0].click();",
            btn
        )

        time.sleep(0.5)

    except Exception as e:
        print(e)

In [ ]:
# 페이지의 전체 텍스트를 가져옴
body = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body[:50000])

In [ ]:
# oracle_df.csv 파일을 읽어옴
# 이 데이터 셋은 oracle 사이트에서 가져온 데이터 셋으로 , 선수들에 대한 정보가 담긴 데이터 셋임
oracle_df = pd.read_csv("oracle_df.csv")

In [ ]:
# 선수들의 이름을 담을 리스트를 만듬 
# 기준은 oracle에 있는 선수들의 이름을 기준으로 함
accounts = []

lines = body.split("\n")

oracle_player_set = set(
    oracle_df['playername'].unique()
)

In [ ]:
i = 0

# 페이지의 텍스트를 한 줄씩 탐색하면서 oracle에 있는 선수들의 이름이 있는지 탐색
while i < len(lines):

    line = lines[i].strip()

    # 선수 발견
    if line in oracle_player_set:

        player = line

        # 이후 15줄만 탐색
        for j in range(i+1, min(i+15, len(lines))):

            next_line = lines[j].strip()

            # 다음 선수 나오면 중단
            if next_line in oracle_player_set:
                break

            # tagline 발견
            if next_line.startswith("#"):

                tagline = next_line.replace("#", "").strip()

                riot_id = lines[j-1].strip()
                # 이렇게 riot id와 tagline을 가져와서 넣음 
                accounts.append({
                    "playername": player,
                    "riot_id": riot_id,
                    "tagline": tagline
                })

    i += 1

In [ ]:
accounts_df = pd.DataFrame(accounts)
    
print(accounts_df.head())

In [ ]:
# 이걸 한번 더 하지 않기 위해 이걸 csv로 저장함
accounts_df.to_csv("accounts_df_KR.csv", index=False)


# RIOT API설정



In [1]:
import requests
import pandas as pd
import dotenv
import os

dotenv.load_dotenv()

riot_api_key = os.getenv("RiotAPIKey")

# Riot Developer Portal에서 발급
API_KEY = riot_api_key

headers = {
    "X-Riot-Token": API_KEY
}

In [2]:
# 예시 소환사명
summoner_name = "Hide on bush"
tag_line = "KR1"

# Riot ID -> PUUID
url = f"https://asia.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{summoner_name}/{tag_line}"

response = requests.get(url, headers=headers)

account_data = response.json()

print(account_data)

puuid = account_data['puuid']
print("PUUID:", puuid)

{'puuid': 'lLj5ZFE5BYnVXkJJim-UKSfy5NO0UkyByxYHvRcLIe1246rN_dnQUcW0WIyR5McqzgruVXODuiQsOg', 'gameName': 'Hide on bush', 'tagLine': 'KR1'}
PUUID: lLj5ZFE5BYnVXkJJim-UKSfy5NO0UkyByxYHvRcLIe1246rN_dnQUcW0WIyR5McqzgruVXODuiQsOg


In [3]:
rank_qu = "https://kr.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
response = requests.get(rank_qu, headers=headers)

challenger_data = response.json()

challenger_data = pd.DataFrame(challenger_data)
challenger_data.head()
print(challenger_data.shape)
print(challenger_data.columns)


(300, 5)
Index(['tier', 'leagueId', 'queue', 'name', 'entries'], dtype='str')


In [4]:
challenger_data

,tier,leagueId,queue,name,entries
0,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'rSs9HiF3cE0JCHM0tK4xFsx8QR9eyBvgYRn...
1,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'Q_5rN7n8B0-8OtKj_hJtpJhrjiPiEt9Juxt...
2,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': '6HyoDnID4EhYPu6cZW_LLNBWtZ4IyhVsWFs...
3,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': '9kF2HSN-IJKnjjl45Vd6W2w7QMTq1lYpz6s...
4,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'CMqC1inIb3bFqPqIuX0c5gf6UV6OCCVSS-F...
...,...,...,...,...,...
295,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'HTmHtqJ70hhnmseRPNMfqvie9ymuCg15VCk...
296,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'VQYlE_beC3Gg7jxu5HY9DRPtjLmMfhx27Rk...
297,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': 'qfTZcYGRdKkB-Mye-ZXFoMaqwRpynlwA9BE...
298,CHALLENGER,582523bb-0d36-396f-819c-ce16b5e0e421,RANKED_SOLO_5x5,Fiora's Pyromancers,{'puuid': '2XhUNd475rrymC-bJfhk3NSFfesmpZ1lpbw...


* 잘 되는지 확인 완료 

# Oracle 선수들의 솔로랭크 매칭 데이터 가져오기 
### 1. Oracle 데이터 선수 추출  

In [5]:
import pandas as pd

# 파일 불러오기
df_2023 = pd.read_csv("2023_clean.csv")
df_2024 = pd.read_csv("2024_clean.csv")
df_2025 = pd.read_csv("2025_clean.csv")
df_2026 = pd.read_csv("2026_clean.csv")

# 시즌 컬럼 추가
df_2023['season'] = 2023
df_2024['season'] = 2024
df_2025['season'] = 2025
df_2026['season'] = 2026

# 전체 통합
oracle_df = pd.concat([
    df_2023,
    df_2024,
    df_2025,
    df_2026
], ignore_index=True)

print(oracle_df.shape)
print(oracle_df.head())

(17660, 40)
  playername   teamname position  patch  year                 date  \
0      Canna  Dplus Kia      top  13.01  2023  2023-01-18 08:17:31   
1     Canyon  Dplus Kia      jng  13.01  2023  2023-01-18 08:17:31   
2  ShowMaker  Dplus Kia      mid  13.01  2023  2023-01-18 08:17:31   
3       Deft  Dplus Kia      bot  13.01  2023  2023-01-18 08:17:31   
4     Kellin  Dplus Kia      sup  13.01  2023  2023-01-18 08:17:31   

   golddiffat15  xpdiffat15  csdiffat15  goldat15  ...     cspm  \
0         724.0       771.0        -4.0    5017.0  ...   8.2633   
1         338.0        -3.0        -6.0    5293.0  ...   5.9735   
2         410.0       -32.0        10.0    5333.0  ...   9.2920   
3        1174.0      1234.0        13.0    5651.0  ...  10.3872   
4         530.0       -47.0        -3.0    3685.0  ...   0.1659   

   damagetotowers  firstbloodkill  firstbloodassist  firstbloodvictim  league  \
0          2153.0             0.0               0.0               0.0     LCK   
1 

In [6]:
oracle_df.columns

Index(['playername', 'teamname', 'position', 'patch', 'year', 'date',
       'golddiffat15', 'xpdiffat15', 'csdiffat15', 'goldat15', 'xpat15',
       'csat15', 'kills', 'deaths', 'assists', 'teamkills', 'dpm',
       'damageshare', 'damagetakenperminute', 'damagemitigatedperminute',
       'visionscore', 'vspm', 'wardsplaced', 'wardskilled', 'wpm', 'wcpm',
       'controlwardsbought', 'earnedgold', 'earned gpm', 'earnedgoldshare',
       'cspm', 'damagetotowers', 'firstbloodkill', 'firstbloodassist',
       'firstbloodvictim', 'league', 'datacompleteness', 'gamelength',
       'result', 'season'],
      dtype='str')

In [8]:
oracle_df['playername'].value_counts()

playername
Aiming       396
Zeus         393
Keria        392
ShowMaker    390
Oner         390
            ... 
Trigger        4
Rahel          3
Guwon          2
Slayer         2
Feisty         1
Name: count, Length: 115, dtype: int64

In [16]:
oracle_df.to_csv("oracle_df.csv", index=False)

In [14]:
pro_players = (
    oracle_df[['playername', 'teamname', 'position']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(pro_players.shape)
print(pro_players.head())

(177, 3)
  playername   teamname position
0      Canna  Dplus Kia      top
1     Canyon  Dplus Kia      jng
2  ShowMaker  Dplus Kia      mid
3       Deft  Dplus Kia      bot
4     Kellin  Dplus Kia      sup


In [18]:
target_roles = ['mid', 'jng']

oracle_target = oracle_df[
    oracle_df['position'].str.lower().isin(target_roles)
].copy()

In [19]:
oracle_target['playername']

1           Canyon
2        ShowMaker
6            Croco
7             FATE
11          Canyon
           ...    
17647       Roamer
17651       GIDEON
17652       Roamer
17656       Raptor
17657        VicLa
Name: playername, Length: 7064, dtype: str

In [20]:
account_df_KR = pd.read_csv("accounts_df_KR.csv")

account_df_KR['playername']

0         Kael
1         Kael
2        Viper
3        Viper
4        Viper
        ...   
215       UmTi
216       UmTi
217       UmTi
218    Grizzly
219    Grizzly
Name: playername, Length: 220, dtype: str

In [24]:
merged_df = pd.merge(
    oracle_target,
    account_df_KR,
    on='playername',
    how='inner'
)

merged_df = merged_df.drop_duplicates(
    subset=['playername', 'riot_id', 'tagline']
)

print(merged_df.shape)
merged_df.head()

(99, 42)


,playername,teamname,position,patch,year,date,golddiffat15,xpdiffat15,csdiffat15,goldat15,...,firstbloodkill,firstbloodassist,firstbloodvictim,league,datacompleteness,gamelength,result,season,riot_id,tagline
0,Canyon,Dplus Kia,jng,13.01,2023,2023-01-18 08:17:31,338.0,-3.0,-6.0,5293.0,...,0.0,0.0,0.0,LCK,complete,1808,1,2023,JUGKlNG,kr
1,Canyon,Dplus Kia,jng,13.01,2023,2023-01-18 08:17:31,338.0,-3.0,-6.0,5293.0,...,0.0,0.0,0.0,LCK,complete,1808,1,2023,그어살,KR1
2,Canyon,Dplus Kia,jng,13.01,2023,2023-01-18 08:17:31,338.0,-3.0,-6.0,5293.0,...,0.0,0.0,0.0,LCK,complete,1808,1,2023,HUANG TONG DAYE,KR1
3,ShowMaker,Dplus Kia,mid,13.01,2023,2023-01-18 08:17:31,410.0,-32.0,10.0,5333.0,...,0.0,0.0,1.0,LCK,complete,1808,1,2023,DK ShowMaker,KR1
4,ShowMaker,Dplus Kia,mid,13.01,2023,2023-01-18 08:17:31,410.0,-32.0,10.0,5333.0,...,0.0,0.0,1.0,LCK,complete,1808,1,2023,DWG KIA,KR1


In [ ]:
# merged_df.to_csv("merged_df.csv", index=False)

In [26]:
merged_df.groupby('playername').size()

playername
Bdd          4
BuLLDoG      1
Calix        3
Canyon       3
Chovy        2
Clid         2
Clozer       3
Croco        3
Cuzz         4
Ellim        2
FATE         3
Faker        1
Fisher       2
GIDEON       2
Grizzly      2
Guwon        2
HamBak       3
Juhan        2
Kanavi       3
Karis        2
Loki         2
Lucid        2
Oner         3
Peanut       4
Pullbae      2
Pyosik       1
Quad         5
Raptor       1
Roamer       2
Scout        3
SeTab        2
ShowMaker    3
Sponge       1
Sylvie       2
Ucal         3
UmTi         4
VicLa        1
Willer       2
YoungJae     2
Zeka         3
kyeahoo      2
dtype: int64

In [28]:
merged_df[
    merged_df['playername'] == 'Faker'
]

# 여기 데이터 셋을 보면 라이엇 id , 선수명만 중요함.
# 여기서의 골드 수급량이나 이런건 의미가 없음.
# 왜냐면 라이엇 api로 가져올 수 있는 데이터는 라이엇 id로 가져와야하기 때문. 
# 선수명은 그냥 매칭하기 위한 용도.

,playername,teamname,position,patch,year,date,golddiffat15,xpdiffat15,csdiffat15,goldat15,...,firstbloodkill,firstbloodassist,firstbloodvictim,league,datacompleteness,gamelength,result,season,riot_id,tagline
27,Faker,T1,mid,13.01,2023,2023-01-18 11:00:07,226.0,367.0,16.0,5309.0,...,0.0,0.0,0.0,LCK,complete,2162,1,2023,Hide on bush,KR1


### account로 puuid추출 

In [4]:
import pandas as pd
merged_df = pd.read_csv("merged_df.csv")

In [6]:
import requests
import pandas as pd
import time


def get_puuid(game_name, tag_line):
    url = f"https://asia.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}"

    headers = {
        "X-Riot-Token": API_KEY
    }

    r = requests.get(url, headers=headers)

    if r.status_code == 200:
        data = r.json()
        return data.get("puuid")

    return None

In [7]:
merged_df['puuid'] = None

for idx, row in merged_df.iterrows():

    puuid = get_puuid(
        row['riot_id'],
        row['tagline']
    )

    merged_df.at[idx, 'puuid'] = puuid

    time.sleep(1.2)

In [8]:
merged_df['puuid'].isna().sum()

np.int64(5)

In [9]:
merged_df[
    merged_df['puuid'].duplicated(keep=False)
]

,playername,teamname,position,patch,year,date,golddiffat15,xpdiffat15,csdiffat15,goldat15,...,firstbloodassist,firstbloodvictim,league,datacompleteness,gamelength,result,season,riot_id,tagline,puuid
11,FATE,DRX,mid,13.01,2023,2023-01-18 08:17:31,-410.0,32.0,-10.0,4923.0,...,1.0,0.0,LCK,complete,1808,0,2023,정재이스,617,None
39,Cuzz,KT Rolster,jng,13.01,2023,2023-01-20 08:11:59,-112.0,153.0,12.0,4550.0,...,0.0,0.0,LCK,complete,1559,1,2023,lililiiilil,KR121,None
50,Clozer,Liiv SANDBOX,mid,13.01,2023,2023-01-20 11:29:11,246.0,618.0,11.0,5419.0,...,0.0,0.0,LCK,complete,1638,0,2023,인천물주먹,인천사람,None
59,Quad,Nongshim RedForce,mid,13.10,2023,2023-06-08 11:00:48,-262.0,664.0,8.0,5004.0,...,0.0,0.0,LCK,complete,1925,0,2023,꼬북칩먹고힘내자,KR1,None
64,HamBak,KT Rolster,jng,13.14,2023,2023-08-06 09:43:51,228.0,361.0,5.0,5222.0,...,0.0,0.0,LCK,complete,2159,1,2023,와하항,kr0,None


* 결측치 제거함 

In [10]:
merged_df = merged_df.dropna(subset=['puuid'])

In [11]:
merged_df.to_csv("merged_df.csv", index=False)

## match ID 가져오기

In [14]:
import requests
import pandas as pd
import time
import dotenv
import os

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

# =========================================================
# match ids 수집 함수 (실전용)
# =========================================================
def get_match_ids(
    puuid,
    total_count=100,
    max_retry=3
):

    # =====================================================
    # puuid 검증
    # =====================================================
    if pd.isna(puuid) or puuid is None or puuid == "":

        print("[INVALID PUUID]")

        return []

    # =====================================================
    # Riot 제한 방어
    # =====================================================
    if total_count > 1000:

        print("[WARNING] 너무 큰 요청 → 1000으로 제한")

        total_count = 1000

    headers = {
        "X-Riot-Token": API_KEY
    }

    all_match_ids = []

    # =====================================================
    # pagination
    # =====================================================
    for start in range(0, total_count, 100):

        count = min(100, total_count - start)

        url = (
            f"https://asia.api.riotgames.com"
            f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
            f"?start={start}&count={count}"
        )

        retry = 0

        while retry < max_retry:

            try:

                r = requests.get(
                    url,
                    headers=headers,
                    timeout=10
                )

                # =========================================
                # 정상 응답
                # =========================================
                if r.status_code == 200:

                    data = r.json()

                    if isinstance(data, list):

                        all_match_ids.extend(data)

                        print(
                            f"[SUCCESS] "
                            f"start={start} "
                            f"/ {len(data)}개"
                        )

                        break

                    else:

                        print("[INVALID RESPONSE]")

                        break

                # =========================================
                # 인증 실패
                # =========================================
                elif r.status_code == 401:

                    print("[401] API KEY 만료 또는 오류")

                    return []

                # =========================================
                # 존재하지 않음
                # =========================================
                elif r.status_code == 404:

                    print(f"[404] NOT FOUND")

                    break

                # =========================================
                # Rate Limit
                # =========================================
                elif r.status_code == 429:

                    retry_after = r.headers.get("Retry-After")

                    wait_time = (
                        int(retry_after)
                        if retry_after
                        else 120
                    )

                    print(
                        f"[429] RATE LIMIT "
                        f"{wait_time}초 대기"
                    )

                    time.sleep(wait_time)

                    retry += 1

                # =========================================
                # 서버 오류
                # =========================================
                elif r.status_code in [500, 502, 503, 504]:

                    print(
                        f"[SERVER ERROR] "
                        f"{r.status_code}"
                    )

                    retry += 1

                    time.sleep(10)

                # =========================================
                # 잘못된 요청
                # =========================================
                elif r.status_code == 400:

                    print(
                        f"[400] BAD REQUEST "
                        f"start={start}, count={count}"
                    )

                    break

                # =========================================
                # 기타 오류
                # =========================================
                else:

                    print(
                        f"[ERROR] STATUS: "
                        f"{r.status_code}"
                    )

                    break

            # =============================================
            # Timeout
            # =============================================
            except requests.exceptions.Timeout:

                print("[TIMEOUT] 재시도")

                retry += 1

                time.sleep(5)

            # =============================================
            # Connection Error
            # =============================================
            except requests.exceptions.ConnectionError:

                print("[CONNECTION ERROR] 재시도")

                retry += 1

                time.sleep(5)

            # =============================================
            # 기타 예외
            # =============================================
            except Exception as e:

                print(f"[EXCEPTION] {e}")

                break

        # 페이지 간 sleep
        time.sleep(1.2)

    # =====================================================
    # 중복 제거
    # =====================================================
    all_match_ids = list(set(all_match_ids))

    return all_match_ids

In [16]:
import os 

CHECKPOINT_DIR = "API_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "match_ids.csv"
)

all_match_ids = []

for idx, row in merged_df.iterrows():

    try:

        playername = row['playername']
        position = row['position']
        puuid = row['puuid']

        # puuid 결측 제거
        if pd.isna(puuid):
            print(f"[SKIP] {playername} - PUUID 없음")
            continue

        # 최근 100게임 수집
        match_ids = get_match_ids(
            puuid=puuid,
            total_count=500
        )

        # 경기 없는 경우
        if len(match_ids) == 0:
            print(f"[NO MATCH] {playername}")
            continue

        # 저장
        for match_id in match_ids:

            all_match_ids.append({
                'playername': playername,
                'position': position,
                'puuid': puuid,
                'match_id': match_id
            })

        print(
            f"[{idx}] {playername} "
            f"- {len(match_ids)}개 수집 완료"
        )

        # =========================================
        # 체크포인트 저장
        # =========================================
        if idx % 5 == 0:

            checkpoint_df = pd.DataFrame(all_match_ids)

            checkpoint_df = checkpoint_df.drop_duplicates(
                subset=['match_id']
            )

            checkpoint_df.to_csv(
                SAVE_PATH,
                index=False,
                encoding='utf-8-sig'
            )

            print(
                f"[CHECKPOINT SAVED] "
                f"{len(checkpoint_df)}개 저장 완료"
            )



        # Rate Limit 방지
        time.sleep(1.5)

    except Exception as e:

        print(f"[LOOP ERROR] {idx} / {e}")

        continue


[SUCCESS] start=0 / 100개
[SUCCESS] start=100 / 100개
[SUCCESS] start=200 / 100개
[SUCCESS] start=300 / 100개
[SUCCESS] start=400 / 75개
[0] Canyon - 475개 수집 완료
[CHECKPOINT SAVED] 475개 저장 완료
[SUCCESS] start=0 / 100개
[SUCCESS] start=100 / 24개
[SUCCESS] start=200 / 0개
[SUCCESS] start=300 / 0개
[SUCCESS] start=400 / 0개
[1] Canyon - 124개 수집 완료
[SUCCESS] start=0 / 0개
[SUCCESS] start=100 / 0개
[SUCCESS] start=200 / 0개
[SUCCESS] start=300 / 0개
[SUCCESS] start=400 / 0개
[NO MATCH] Canyon
[SUCCESS] start=0 / 100개
[SUCCESS] start=100 / 100개
[SUCCESS] start=200 / 100개
[SUCCESS] start=300 / 100개
[SUCCESS] start=400 / 100개
[3] ShowMaker - 500개 수집 완료
[SUCCESS] start=0 / 37개
[SUCCESS] start=100 / 0개
[SUCCESS] start=200 / 0개
[SUCCESS] start=300 / 0개
[SUCCESS] start=400 / 0개
[4] ShowMaker - 37개 수집 완료
[SUCCESS] start=0 / 100개
[SUCCESS] start=100 / 100개
[SUCCESS] start=200 / 100개
[SUCCESS] start=300 / 11개
[SUCCESS] start=400 / 0개
[5] ShowMaker - 311개 수집 완료
[CHECKPOINT SAVED] 1420개 저장 완료
[SUCCESS] start=0 / 100개


In [17]:
# =========================================================
# DataFrame 변환
# =========================================================

match_df = pd.DataFrame(all_match_ids)

# 중복 제거
match_df = match_df.drop_duplicates(
    subset=['match_id']
)

# 저장
match_df.to_csv(
    "match_ids.csv",
    index=False,
    encoding='utf-8-sig'
)

print("=" * 50)
print("최종 match 수:", len(match_df))
print("=" * 50)

최종 match 수: 25212


* 각자 데이터 수가 다른 이유는 계정이 여러개라 그런거임 

In [18]:
match_df.groupby('playername').size()   

playername
Bdd           170
BuLLDoG       464
Calix         653
Canyon        599
Chovy         420
Clid          941
Clozer        428
Croco        1397
Cuzz          953
Ellim         486
FATE          995
Faker         411
Fisher        668
GIDEON        919
Grizzly       451
Guwon         405
HamBak        844
Juhan         906
Kanavi        438
Karis         452
Loki          719
Lucid         383
Oner          483
Peanut        462
Pullbae       501
Pyosik        460
Quad          472
Raptor        376
Roamer        324
Scout         399
SeTab         877
ShowMaker     821
Sponge        383
Sylvie        899
Ucal          825
UmTi          561
VicLa         317
Willer        690
YoungJae      830
Zeka          598
kyeahoo       832
dtype: int64

In [19]:
print("중복 제거 전:", match_df.shape)

print(
    "중복 수:",
    match_df['match_id'].duplicated().sum()
)

match_df = match_df.drop_duplicates(
    subset=['match_id']
)

print("중복 제거 후:", match_df.shape)

중복 제거 전: (25212, 4)
중복 수: 0
중복 제거 후: (25212, 4)


In [20]:
match_df.groupby('playername')['match_id'].nunique()

playername
Bdd           170
BuLLDoG       464
Calix         653
Canyon        599
Chovy         420
Clid          941
Clozer        428
Croco        1397
Cuzz          953
Ellim         486
FATE          995
Faker         411
Fisher        668
GIDEON        919
Grizzly       451
Guwon         405
HamBak        844
Juhan         906
Kanavi        438
Karis         452
Loki          719
Lucid         383
Oner          483
Peanut        462
Pullbae       501
Pyosik        460
Quad          472
Raptor        376
Roamer        324
Scout         399
SeTab         877
ShowMaker     821
Sponge        383
Sylvie        899
Ucal          825
UmTi          561
VicLa         317
Willer        690
YoungJae      830
Zeka          598
kyeahoo       832
Name: match_id, dtype: int64

In [21]:
len(match_df['match_id'].unique()) 

25212

* 이제 선수 단위로 분석 시작함 
* match ID를 기준으로 API req 보내기 

### match Detail 가져오기 

In [1]:
# 파일 불러오기
import pandas as pd

match_df = pd.read_csv("match_ids.csv")

import os
import dotenv

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

In [2]:
import requests
import time

def get_match_detail(
    match_id,
    max_retry=3
):

    url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/{match_id}"
    )

    headers = {
        "X-Riot-Token": API_KEY
    }

    retry = 0

    while retry < max_retry:

        try:

            r = requests.get(
                url,
                headers=headers,
                timeout=10
            )

            # =====================================
            # 성공
            # =====================================
            if r.status_code == 200:

                return r.json()

            # =====================================
            # Rate Limit
            # =====================================
            elif r.status_code == 429:

                retry_after = r.headers.get(
                    "Retry-After"
                )

                wait_time = (
                    int(retry_after)
                    if retry_after
                    else 120
                )

                print(
                    f"[429] {wait_time}초 대기"
                )

                time.sleep(wait_time)

                retry += 1

            # =====================================
            # 인증 오류
            # =====================================
            elif r.status_code == 401:

                print("[401] API KEY 오류")

                return None

            # =====================================
            # 존재하지 않음
            # =====================================
            elif r.status_code == 404:

                print(f"[404] {match_id}")

                return None

            # =====================================
            # 서버 오류
            # =====================================
            elif r.status_code in [500, 502, 503, 504]:

                print(
                    f"[SERVER ERROR] "
                    f"{r.status_code}"
                )

                retry += 1

                time.sleep(10)

            else:

                print(
                    f"[ERROR] {r.status_code}"
                )

                return None

        except Exception as e:

            print(f"[EXCEPTION] {e}")

            retry += 1

            time.sleep(5)

    print(f"[FAILED] {match_id}")

    return None

<!-- match_data['info']['participants'] -->

In [7]:
test_match_id = match_df.iloc[0]['match_id']

test_data = get_match_detail(test_match_id)

print(test_data.keys())

dict_keys(['metadata', 'info'])


In [8]:
participants = test_data['info']['participants']

print(len(participants))

10


In [9]:
# 잘되나만 하나만 확인하는거 선수 아닐 가능성 높음 
participants = test_data['info']['participants']

participants[0]

{'PlayerBehavior': {'PlayerBehavior_IsHeroInCombat': 0},
 'PlayerScore0': 0,
 'PlayerScore1': 0,
 'PlayerScore10': 0,
 'PlayerScore11': 0,
 'PlayerScore2': 0,
 'PlayerScore3': 0,
 'PlayerScore4': 0,
 'PlayerScore5': 0,
 'PlayerScore6': 0,
 'PlayerScore7': 0,
 'PlayerScore8': 0,
 'PlayerScore9': 0,
 'allInPings': 0,
 'assistMePings': 0,
 'assists': 1,
 'baronKills': 0,
 'basicPings': 0,
 'challenges': {'12AssistStreakCount': 0,
  'HealFromMapSources': 0,
  'InfernalScalePickup': 0,
  'SWARM_DefeatAatrox': 0,
  'SWARM_DefeatBriar': 0,
  'SWARM_DefeatMiniBosses': 0,
  'SWARM_EvolveWeapon': 0,
  'SWARM_Have3Passives': 0,
  'SWARM_KillEnemy': 0,
  'SWARM_PickupGold': 0,
  'SWARM_ReachLevel50': 0,
  'SWARM_Survive15Min': 0,
  'SWARM_WinWith5EvolvedWeapons': 0,
  'abilityUses': 96,
  'acesBefore15Minutes': 0,
  'alliedJungleMonsterKills': 0,
  'baronTakedowns': 0,
  'blastConeOppositeOpponentCount': 0,
  'bountyGold': 0,
  'buffsStolen': 0,
  'completeSupportQuestInTime': 0,
  'controlWardsPl

## match data API로 가져오기 
* "그 10명 중에서 내 프로 선수 puuid와 일치하는 participant만 추출"

## match data 파싱 
match_data['info']['participants']
-> 대충 이런식으로 10명의 정보가 다 들어있는데 
-> 여기서 선수들의 퍼포먼스만 볼 예정임 
-> 프로선수 participant만 추출 예정 

* 수집할 컬럼 이름 

| Oracle            | Riot match-v5                   |
| ----------------- | ------------------------------- |
| kills             | kills                           |
| deaths            | deaths                          |
| assists           | assists                         |
| visionscore       | visionScore                     |
| earnedgold        | goldEarned                      |
| cspm              | totalMinionsKilled / gameLength |
| damagetochampions | totalDamageDealtToChampions     |
| damagetaken       | totalDamageTaken                |
| wardsplaced       | wardsPlaced                     |
| wardskilled       | wardsKilled                     |


In [ ]:
# def extract_player_data(match_data, puuid):

#     participants = match_data['info']['participants']

#     game_length = match_data['info']['gameDuration'] / 60

#     for p in participants:

#         if p['puuid'] == puuid:

#             return {

#                 'match_id': match_data['metadata']['matchId'],

#                 'champion': p['championName'],

#                 'kills': p['kills'],
#                 'deaths': p['deaths'],
#                 'assists': p['assists'],

#                 'visionScore': p['visionScore'],

#                 'goldEarned': p['goldEarned'],

#                 'totalDamageDealtToChampions':
#                     p['totalDamageDealtToChampions'],

#                 'totalDamageTaken':
#                     p['totalDamageTaken'],

#                 'wardsPlaced':
#                     p['wardsPlaced'],

#                 'wardsKilled':
#                     p['wardsKilled'],

#                 'cs':
#                     p['totalMinionsKilled'] +
#                     p['neutralMinionsKilled'],

#                 'cspm':
#                     (
#                         p['totalMinionsKilled'] +
#                         p['neutralMinionsKilled']
#                     ) / game_length,

#                 'win': p['win']
#             }

#     return None

In [3]:
def extract_player_data(match_data, puuid):

    participants = match_data['info']['participants']

    valid_positions = {
        'TOP',
        'JUNGLE',
        'MIDDLE',
        'BOTTOM',
        'UTILITY'
    }

    for p in participants:

        if p['puuid'] == puuid:

            team_pos = p.get('teamPosition')
            indiv_pos = p.get('individualPosition')

            # 정상 포지션만
            if team_pos not in valid_positions:
                return None

            # 포지션 불일치 제거
            if team_pos != indiv_pos:
                return None

            game_duration = (
                match_data['info']['gameDuration']
            )

            # 리메이크 제거
            if game_duration < 600:
                return None

            # 총 CS
            cs = (
                p['totalMinionsKilled']
                + p['neutralMinionsKilled']
            )

            # 반환 데이터
            filtered_p = {

                # -------------------
                # 기본 정보
                # -------------------

                'match_id':
                    match_data['metadata']['matchId'],

                'puuid':
                    p['puuid'],

                'summonerName':
                    p['summonerName'],

                'riotIdGameName':
                    p.get('riotIdGameName'),

                'championName':
                    p['championName'],

                'teamPosition':
                    p['teamPosition'],

                'win':
                    p['win'],

                'gameDuration':
                    game_duration,

                # -------------------
                # 전투
                # -------------------

                'kills':
                    p['kills'],

                'deaths':
                    p['deaths'],

                'assists':
                    p['assists'],

                'doubleKills':
                    p['doubleKills'],

                'tripleKills':
                    p['tripleKills'],

                'quadraKills':
                    p['quadraKills'],

                'pentaKills':
                    p['pentaKills'],

                # -------------------
                # 데미지
                # -------------------

                'totalDamageDealtToChampions':
                    p['totalDamageDealtToChampions'],

                'totalDamageTaken':
                    p['totalDamageTaken'],

                'damageSelfMitigated':
                    p['damageSelfMitigated'],

                'timeCCingOthers':
                    p['timeCCingOthers'],

                # -------------------
                # 시야
                # -------------------

                'visionScore':
                    p['visionScore'],

                'wardsPlaced':
                    p['wardsPlaced'],

                'wardsKilled':
                    p['wardsKilled'],

                'visionWardsBoughtInGame':
                    p['visionWardsBoughtInGame'],

                # -------------------
                # 성장
                # -------------------

                'goldEarned':
                    p['goldEarned'],

                'champLevel':
                    p['champLevel'],

                'cs':
                    cs,

                # -------------------
                # 오브젝트
                # -------------------

                'damageDealtToObjectives':
                    p['damageDealtToObjectives'],

                'damageDealtToTurrets':
                    p['damageDealtToTurrets'],

                'dragonKills':
                    p['dragonKills'],

                'baronKills':
                    p['baronKills'],

                'turretKills':
                    p['turretKills'],

                # -------------------
                # 퍼블
                # -------------------

                'firstBloodKill':
                    p['firstBloodKill'],

                'firstBloodAssist':
                    p['firstBloodAssist'],
            }

            # -------------------
            # 파생 변수
            # -------------------

            minutes = game_duration / 60

            filtered_p['kda'] = (
                (p['kills'] + p['assists'])
                / (p['deaths'] + 1)
            )

            filtered_p['cspm'] = (
                cs / minutes
            )

            filtered_p['dpm'] = (
                p['totalDamageDealtToChampions']
                / minutes
            )

            filtered_p['earned_gpm'] = (
                p['goldEarned']
                / minutes
            )

            filtered_p['vspm'] = (
                p['visionScore']
                / minutes
            )

            filtered_p['wpm'] = (
                p['wardsPlaced']
                / minutes
            )

            filtered_p['wcpm'] = (
                p['wardsKilled']
                / minutes
            )

            filtered_p['damagetakenperminute'] = (
                p['totalDamageTaken']
                / minutes
            )

            filtered_p['damagemitigatedperminute'] = (
                p['damageSelfMitigated']
                / minutes
            )

            return filtered_p

    return None

In [31]:
# # 테스트용 1개 선택
# test_row = match_df.iloc[0]

# test_match_id = test_row['match_id']
# test_puuid = test_row['puuid']

# print("TEST MATCH:", test_match_id)

# # match detail 가져오기
# match_data = get_match_detail(test_match_id)

# # participant 추출
# player_data = extract_player_data(
#     match_data,
#     test_puuid
# )

# print(player_data)

TEST MATCH: KR_8186888450
{'match_id': 'KR_8186888450', 'champion': 'Ambessa', 'kills': 0, 'deaths': 2, 'assists': 0, 'visionScore': 20, 'goldEarned': 5218, 'totalDamageDealtToChampions': 2213, 'totalDamageTaken': 11159, 'wardsPlaced': 7, 'wardsKilled': 4, 'cs': 123, 'cspm': 8.047982551799345, 'win': False}


In [5]:
import os
import pandas as pd
import time

# =====================================================
# 저장 경로
# =====================================================

CHECKPOINT_DIR = "API_checkpoint"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "player포지션별_pull_match_data.csv"
)

# =====================================================
# 기존 데이터 불러오기 (복구용)
# =====================================================

if os.path.exists(SAVE_PATH):

    existing_df = pd.read_csv(SAVE_PATH)

    all_player_data = existing_df.to_dict('records')

    collected_match_ids = set(
        existing_df['match_id'].unique()
    )

    print(
        f"기존 데이터 로드 완료 "
        f"({len(collected_match_ids)}개)"
    )

else:

    all_player_data = []

    collected_match_ids = set()

    print("새로운 수집 시작")

# =====================================================
# match detail 수집
# =====================================================

for idx, row in match_df.iterrows():

    try:

        match_id = row['match_id']
        puuid = row['puuid']
        playername = row['playername']

        # =============================================
        # 이미 수집한 경기 skip
        # =============================================
        if match_id in collected_match_ids:

            print(f"[SKIP] {match_id}")

            continue

        # =============================================
        # match detail 가져오기
        # =============================================
        match_data = get_match_detail(match_id)

        if match_data is None:

            print(f"[NO DATA] {match_id}")

            continue

        # =============================================
        # participant 추출
        # =============================================
        player_data = extract_player_data(
            match_data,
            puuid
        )

        if player_data is None:

            print(
                f"[NO PARTICIPANT] "
                f"{playername}"
            )

            continue

        # =============================================
        # 추가 정보 저장
        # =============================================
        player_data['playername'] = playername
        player_data['puuid'] = puuid

        all_player_data.append(player_data)

        collected_match_ids.add(match_id)

        print(
            f"[{idx}] "
            f"{playername} "
            f"- 저장 완료"
        )

        # =============================================
        # 체크포인트 저장
        # =============================================
        if idx % 50 == 0:

            checkpoint_df = pd.DataFrame(
                all_player_data
            )

            checkpoint_df.to_csv(
                SAVE_PATH,
                index=False,
                encoding='utf-8-sig'
            )

            print(
                f"[CHECKPOINT] "
                f"{len(checkpoint_df)}개 저장"
            )

        # =============================================
        # Rate limit 방지
        # =============================================
        # time.sleep(1.2)

    except Exception as e:

        print(
            f"[ERROR] "
            f"{idx} / {e}"
        )

        continue

# =====================================================
# 최종 저장
# =====================================================

final_df = pd.DataFrame(all_player_data)

final_df.to_csv(
    SAVE_PATH,
    index=False,
    encoding='utf-8-sig'
)

print("=" * 50)
print("최종 저장 완료")
print("총 데이터 수:", len(final_df))
print("=" * 50)

새로운 수집 시작
[0] Canyon - 저장 완료
[CHECKPOINT] 1개 저장
[1] Canyon - 저장 완료
[2] Canyon - 저장 완료
[3] Canyon - 저장 완료
[4] Canyon - 저장 완료
[5] Canyon - 저장 완료
[6] Canyon - 저장 완료
[7] Canyon - 저장 완료
[8] Canyon - 저장 완료
[9] Canyon - 저장 완료
[10] Canyon - 저장 완료
[11] Canyon - 저장 완료
[12] Canyon - 저장 완료
[13] Canyon - 저장 완료
[NO PARTICIPANT] Canyon
[15] Canyon - 저장 완료
[16] Canyon - 저장 완료
[17] Canyon - 저장 완료
[18] Canyon - 저장 완료
[19] Canyon - 저장 완료
[NO PARTICIPANT] Canyon
[21] Canyon - 저장 완료
[22] Canyon - 저장 완료
[NO PARTICIPANT] Canyon
[24] Canyon - 저장 완료
[25] Canyon - 저장 완료
[26] Canyon - 저장 완료
[27] Canyon - 저장 완료
[28] Canyon - 저장 완료
[NO PARTICIPANT] Canyon
[30] Canyon - 저장 완료
[31] Canyon - 저장 완료
[32] Canyon - 저장 완료
[33] Canyon - 저장 완료
[34] Canyon - 저장 완료
[35] Canyon - 저장 완료
[NO PARTICIPANT] Canyon
[37] Canyon - 저장 완료
[38] Canyon - 저장 완료
[39] Canyon - 저장 완료
[40] Canyon - 저장 완료
[41] Canyon - 저장 완료
[42] Canyon - 저장 완료
[43] Canyon - 저장 완료
[44] Canyon - 저장 완료
[45] Canyon - 저장 완료
[46] Canyon - 저장 완료
[47] Canyon - 저장 완료
[

* 중간 저장할때마다 곂치는게 존재함으로 중복된 데이터에 대한 처리를 해줘야함 
* 하지만 한 게임에 같은 선수들이 있을 수 있다보니 생기는 문제임 
* 같은 경기(match_id)에 프로 여러 명이 있었던 경우
* match_id 가 곂치는 부분은 "프로 간 같은 게임 존재"하는걸로 해석을 할 수 있기에 이 값은 처리를 안하는게 맞음

* `그래도 완전 동일 row 중복이나 , 같은 게임에 플레이어 이름이 2번 들어간 경우는 오류임으로 제거해줌 `

In [6]:
df = pd.read_csv("API_checkpoint/player포지션별_pull_match_data.csv")
print(df.shape)

(22535, 43)


In [7]:
print("원본 데이터 크기:", df.shape)

# 1. 완전 동일 row 중복 확인
full_duplicates = df.duplicated().sum()

print("완전 동일 row 중복 수:", full_duplicates)

# 2. 같은 선수 + 같은 경기 중복 확인
player_match_duplicates = df.duplicated(
    subset=['playername', 'match_id']
).sum()

print(
    "playername + match_id 중복 수:",
    player_match_duplicates
)

# ============================================
# 중복 제거
# ============================================

df_clean = df.drop_duplicates(
    subset=['playername', 'match_id']
)

print("중복 제거 후 데이터 크기:", df_clean.shape)

# ============================================
# 제거된 row 수 확인
# ============================================

removed_rows = len(df) - len(df_clean)

print("제거된 row 수:", removed_rows)

원본 데이터 크기: (22535, 43)
완전 동일 row 중복 수: 0
playername + match_id 중복 수: 0
중복 제거 후 데이터 크기: (22535, 43)
제거된 row 수: 0
